[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanvykopal/nlp-kinit-2026/blob/main/examples/rl/gspo.ipynb)

# Reinforcement Learning with GSPO on Tower of Hanoi

The FFT/LoRA notebooks fit the *demonstrations* — supervised fine-tuning
(SFT): training to reproduce example answers — and one in five of those
was corrupted — a single move deliberately swapped, replaced, deleted or
inserted. Both methods inherited that per-move error rate and stalled at
the same place. GSPO never looks at a target: for each prompt it samples
a **group** of completions, scores each by *running it through the
verifier*, and pushes probability mass toward the ones that scored above
the group average. The reward is ground truth, not a noisy demonstration,
so the ceiling moves.

We run it twice: cold-start from the base model (expected to fail — the
single most common way RL runs die), then from the FFT checkpoint.

Runtime: ~18 minutes on a free Colab T4. Run the FFT notebook first.

## 0. Setup

Install the dependencies (minimum versions these notebooks were validated against).

In [ ]:
get_ipython().system('pip install -q "transformers>=4.56,<5" "trl>=0.24" "peft>=0.14" "datasets>=3.0" accelerate matplotlib pandas')

In [ ]:
get_ipython().system('git clone --depth 1 https://github.com/ivanvykopal/nlp-kinit-2026')
import sys; sys.path.insert(0, "nlp-kinit-2026")

from lab import evaluation, generation, plotting, probe, report

## Shared code

The cells below define the constants and the Tower of Hanoi
task logic (`tasks/hanoi.py`) directly in the notebook's own namespace —
run them once from the top; everything after uses these names without any
`import`. Generic infrastructure (batched generation, evaluation,
reporting, plotting) is *imported for real* in the cell above, from
`lab/`, which never mentions Hanoi by name.


In [ ]:
"""Generic constants shared by every notebook: model, dirs, token/training budgets.

No task-shaped constants here — those live in the task module (see
`tasks/hanoi.py`'s `PROBE_GROUP_VALUES`).
"""
from pathlib import Path

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
SEED = 42

Every notebook reads the same dataset from the HuggingFace Hub. It has five
named splits: `train` (answers to imitate, one in five deliberately wrong),
`grpo_train` (puzzles only, no answers — the split name predates our rename
to GSPO), `heldout` (the number we care about), `train_instances` (a
memorisation check) and `extrapolation` (bigger puzzles than any in
training).

In [ ]:
DATASET_REPO = "ivykopal/nlp-kinit-2026-hanoi"

Notebooks run from different working directories — `examples/rl/` locally,
`/content` on Colab — so a bare `Path("results")` would point somewhere
different in each one, and the GSPO notebook would fail to find the
checkpoint the FFT notebook saved. This walks up from the current directory
to find the one folder holding both `lab/` and `config.py`.

In [ ]:
def _repo_root() -> Path:
    import sys
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates += [Path(p) for p in sys.path if p]
    for c in candidates:
        if c.is_dir() and (c / "lab").is_dir() and (c / "config.py").is_file():
            return c
    return Path.cwd()  # fallback: original <cwd>/results behaviour


RESULTS_DIR = _repo_root() / "results"

How many tokens we let the model read and write. `MAX_SEQ_LENGTH` caps
training examples; `EVAL_MAX_NEW_TOKENS` has to be generous enough for a
31-move answer, or a correct solution gets cut off and scored as a failure.

In [ ]:
MAX_SEQ_LENGTH = 384
EVAL_MAX_NEW_TOKENS = 576
EVAL_BATCH_SIZE = 16

### BUDGETS: GSPO

`GSPO_MAX_STEPS` uses 45 steps.
A round is `GSPO_BATCH_SIZE * GSPO_GRAD_ACCUM * GSPO_NUM_GENERATIONS` =
4 * 4 * 8 = 128 rollouts. At `num_iterations=1`, 45 optimizer steps is
45 rounds.

In [ ]:
GSPO_MAX_COMPLETION_LENGTH = 256
GSPO_MAX_STEPS = 45
GSPO_COLD_START_MAX_STEPS = 15
GSPO_BATCH_SIZE = 4
GSPO_GRAD_ACCUM = 4
GSPO_NUM_GENERATIONS = 8
GSPO_LEARNING_RATE = 3e-6
GSPO_PROBE_EVERY = 20

## Task: Tower of Hanoi

This module is the one place that knows what "Tower of Hanoi" means:
how to parse a model's answer, how to check whether a sequence of moves
actually solves the puzzle, how to turn a dataset row into a training
example, and how to reward a rollout. Everything in `lab/` is generic —
it never mentions a peg or a disk by name — and depends only on the
functions below.

Swapping this tutorial to a different verifiable task (Sudoku, a graph
coloring problem, anything with a checker) means writing a new module
shaped like this one; `lab/` would not change at all.

A move is a plain `(source, target)` tuple of peg names — `("A", "C")`
reads as "move the top disk of A onto C". No `Move` class: a tuple is
already immutable, printable, and comparable, and one regex below is the
entire move syntax.

In [ ]:
"""Tower of Hanoi: parsing, solving, replaying, scoring, and reward."""
from __future__ import annotations

import random
import re
from typing import List

PEG_NAMES = ["A", "B", "C"]

# The whole move syntax: `A->C`, with optional spaces around the arrow.
MOVE_RE = re.compile(r"([{pegs}])\s*->\s*([{pegs}])".format(pegs="".join(PEG_NAMES)))

Two tiny helpers turn a move tuple into the text form the model reads and
writes, and back — everything else in this module works with `(source,
target)` tuples, so the text form only exists at the model boundary.

In [ ]:
def move_to_text(move) -> str:
    return "{}->{}".format(*move)


def moves_to_text(moves) -> str:
    return "\n".join(move_to_text(m) for m in moves)

Turning model output back into moves needs two different levels of
strictness. `parse_output` is used for scoring a whole flat solution: it
walks every line and keeps a strict per-line count, so junk is counted
rather than silently dropped. `first_move_in` is used by the per-step
formulation, where the model is asked for exactly one move: it just finds
the first well-formed move anywhere in the text and ignores the rest.

In [ ]:
def parse_output(text: str):
    """Parse a whole completion into moves.

    Returns `(moves, n_lines, n_unparsed)`: every non-empty line is either a
    move or counted as unparseable — junk is counted, never silently
    dropped, so the caller can tell a clean list of moves from noisy output.
    """
    moves, n_lines, n_unparsed = [], 0, 0
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        n_lines += 1
        match = MOVE_RE.fullmatch(line)
        if match:
            moves.append(match.groups())
        else:
            n_unparsed += 1
    return moves, n_lines, n_unparsed

`first_move_in` is the lenient counterpart used by the per-step
formulation. `completion_text` exists because a TRL completion arrives as
either raw text or a list of chat turns, and every caller below wants
plain text.

In [ ]:
def first_move_in(text: str):
    """Lenient single-move parser: the first well-formed move anywhere in the text."""
    match = MOVE_RE.search(text)
    return match.groups() if match else None


def completion_text(completion) -> str:
    """A TRL completion is either raw text or a list of chat turns; flatten it."""
    if isinstance(completion, str):
        return completion
    return "".join(turn.get("content") or "" for turn in completion)

A reference solver, used two ways below: to build the optimal
demonstrations the SFT notebooks train on, and to know the optimal move
count for scoring — `2**n - 1`, the closed form for this recursion.

In [ ]:
def solve_hanoi(n_disks, source, auxiliary, target) -> List[tuple]:
    """The unique optimal solution: move n-1 aside, move the bottom disk, move them back."""
    if n_disks == 0:
        return []
    return (
        solve_hanoi(n_disks - 1, source, target, auxiliary)
        + [(source, target)]
        + solve_hanoi(n_disks - 1, auxiliary, source, target)
    )


def optimal_move_count(n_disks: int) -> int:
    return 2 ** n_disks - 1

### The verifier

There is no environment *class* here, and no simulator either. A board is
just a `dict` of `{peg: [disks]}`, top disk last in the list; `apply_move`
is the single stepping primitive (used by the flat replay below *and* by
the per-step formulation further down); and `replay` folds it over a list
of moves. An illegal move ends the attempt — whatever follows it is not
played, so it cannot count for anything.

In [ ]:
def new_pegs(n_disks, source, auxiliary, target) -> dict:
    """All disks stacked on `source`; the other two pegs start empty."""
    return {source: list(range(n_disks, 0, -1)), auxiliary: [], target: []}


def is_legal(pegs: dict, move) -> bool:
    """Legal iff it moves the top disk of one peg onto a bigger (or empty) peg."""
    source, target = move
    if source == target or not pegs[source]:
        return False
    return not pegs[target] or pegs[source][-1] < pegs[target][-1]

`apply_move` is the single stepping primitive — used by the flat `replay`
below.

In [ ]:
def apply_move(pegs: dict, move) -> "tuple[dict, bool]":
    """Legality-checked step. Returns (new_pegs, legal); `pegs` is never mutated."""
    if not is_legal(pegs, move):
        return pegs, False
    source, target = move
    stepped = {peg: list(stack) for peg, stack in pegs.items()}
    stepped[target].append(stepped[source].pop())
    return stepped, True


def is_solved(pegs: dict, n_disks: int, target: str) -> bool:
    return len(pegs[target]) == n_disks

`replay` folds `apply_move` over a whole list of moves, for scoring a flat
solution. An illegal move ends the attempt — whatever follows it is not
played, so it cannot count for anything.

In [ ]:
def replay(moves, n_disks, source, auxiliary, target) -> "tuple[dict, bool]":
    """Play `moves` from the start, stopping at the first illegal move.

    Returns the board it reached and whether an illegal move ended it.
    """
    pegs = new_pegs(n_disks, source, auxiliary, target)
    for move in moves:
        pegs, legal = apply_move(pegs, move)
        if not legal:
            return pegs, True
    return pegs, False

### Loading the dataset

The dataset lives on the HuggingFace Hub as five named splits — `train`,
`grpo_train`, `heldout`, `train_instances`, `extrapolation`. One row looks
like this:

```json
{
  "prompt": "...", "n_disks": 3, "source": "A", "auxiliary": "B", "target": "C",
  "target_response": "...", "optimal_response": "...",
  "corrupted": false, "corruption": null
}
```

In [ ]:
def load_split(name: str, repo_id: str):
    """Load one named split of the dataset from the HuggingFace Hub.

    Returns a `datasets.Dataset` — already iterable/indexable like a list of
    dicts (for `sum(r["corrupted"] for r in rows)`-style inspection) and
    already has `.map()` (for building the SFT/GSPO training format), so
    there's no separate "plain rows" vs. "Dataset" loading step.
    """
    from datasets import load_dataset

    return load_dataset(repo_id, split=name)

### Grouping for the by-group breakdown and the mid-training probe

Hanoi's natural grouping is disk count. A different task might group by
difficulty, grid size, or not at all — `lab.evaluation`/`lab.probe` accept
`group_key=None` and simply skip the breakdown when a task has nothing to
group by.

In [ ]:
def PROBE_GROUP_KEY(sample: dict) -> int:
    return sample["n_disks"]


PROBE_GROUP_VALUES = [4]

### `compute_stats`: what a model's answer scores as

`compute_stats` is what evaluation reports, and it deliberately reports
only three things: **did it solve the puzzle**, **how many moves it wrote
against the 2**n - 1 optimum**, and **was there anything in the output
that was not a move**. Nothing else — every extra field is one more column
to explain and one more thing that can quietly disagree with `solved`.

`"solved"` (a bool) is the only field `lab/` requires; the rest is
free-form, and `lab.evaluation.aggregate` averages whatever numeric/bool
fields it finds without needing to know their names.

In [ ]:
def compute_stats(prediction: str, sample: dict) -> dict:
    """Replay one completion and report what evaluation shows."""
    moves, n_lines, n_unparsed = parse_output(prediction)
    pegs, _illegal = replay(moves, sample["n_disks"], sample["source"],
                            sample["auxiliary"], sample["target"])
    return {
        "solved": is_solved(pegs, sample["n_disks"], sample["target"]),
        "total_moves": len(moves),
        "optimal_moves": optimal_move_count(sample["n_disks"]),
        "unparsed": n_unparsed > 0,
    }

### From dataset row to training example

Three small functions turn one dataset row into what a trainer expects: a
prompt, and — for the two supervised methods — the answer to train on.

In [ ]:
DEFAULT_SYSTEM_PROMPT = (
    "You are an expert algorithmic problem solver. "
    "Solve the Tower of Hanoi puzzle optimally. "
    "Return ONLY one move per line in the format 'A->C'. "
    "Do not provide any explanation."
)

`to_chat_prompt` wraps `DEFAULT_SYSTEM_PROMPT` and the puzzle's own prompt
text into the system+user half of a conversation. `to_gspo_format` carries the puzzle's
coordinates instead of any target, since GSPO never sees a correct answer —
only enough information to check one.

In [ ]:
def to_chat_prompt(sample: dict) -> List[dict]:
    """The prompt half of the conversation (system + user)."""
    return [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
        {"role": "user", "content": sample["prompt"]},
    ]

def to_gspo_format(sample: dict) -> dict:
    """GSPO conversational prompt plus the columns the reward function needs."""
    return {
        "prompt": to_chat_prompt(sample),
        "n_disks": sample["n_disks"],
        "source": sample["source"],
        "auxiliary": sample["auxiliary"],
        "target": sample["target"],
    }

## Reward (GSPO only)

Three components, in a total range of `[-0.7, +2.0]`, logged separately by
TRL so a stalled run shows which one stopped improving. None of them grows
with the number of moves produced — that is the reward-hacking guard from
the theory section: a reward of the form `+1 per legal move` is maximized
by shuffling disks legally forever and never solving anything, and that
policy is perfectly stable, because TRL trains on truncated rollouts by
default.

Before trusting this reward, rank a handful of hand-written trajectories
by hand — an empty answer should score worse than illegal moves, illegal
moves worse than partial progress, and "solved but rambling" worse than
"solved optimally." `tests/tasks/test_hanoi_reward.py` pins exactly this
ordering.

In [ ]:
FORMAT_WEIGHT = 0.2
EMPTY_PENALTY = 0.3
PROGRESS_WEIGHT = 0.5
ILLEGAL_PENALTY = 0.25  # below EMPTY_PENALTY: a wrong attempt beats refusing to answer
SOLVED_BONUS = 1.0
OPTIMAL_BONUS = 0.5
OVERSHOOT_WEIGHT = 0.15
# How many multiples of the optimal move count are still "wasteful" rather
# than "rambling".
OVERSHOOT_TOLERANCE = 3.0

Partial credit needs to know *how far* the model got, which is more than
`compute_stats` reports, so the reward does its own replay here and keeps
those extra numbers out of the evaluation table.

In [ ]:
def reward_stats(prediction: str, sample: dict) -> dict:
    """Everything the reward needs: `compute_stats` plus how far the model got."""
    n_disks = sample["n_disks"]
    moves, n_lines, n_unparsed = parse_output(prediction)
    pegs, illegal = replay(moves, n_disks, sample["source"], sample["auxiliary"],
                           sample["target"])

    placed = 0
    for index, disk in enumerate(pegs[sample["target"]]):
        if disk != n_disks - index:  # correctly stacked, counted from the bottom
            break
        placed += 1

    return {
        "n_lines": n_lines,
        "n_unparsed": n_unparsed,
        "illegal": illegal,
        "progress_fraction": placed / n_disks,
        "solved": placed == n_disks,
        "total_moves": len(moves),
        "optimal_moves": optimal_move_count(n_disks),
    }

Two dense components, checked with the reward as a whole: `format_score`
penalizes noise, `progress_score` gives partial credit for disks placed
minus a penalty for an illegal move.

In [ ]:
def format_score(stats: dict) -> float:
    """Penalize noise; saying nothing at all is the worst option."""
    if stats["n_lines"] == 0:
        return -EMPTY_PENALTY
    return -FORMAT_WEIGHT * stats["n_unparsed"] / stats["n_lines"]


def progress_score(stats: dict) -> float:
    """Dense partial credit for disks placed, minus a flat penalty for slipping."""
    score = PROGRESS_WEIGHT * stats["progress_fraction"]
    if stats["illegal"]:
        score -= ILLEGAL_PENALTY
    return score

`solved_score` is the sparse half of the reward: nothing until the puzzle
is actually solved, then a bonus, plus a small penalty for rambling well
past the optimal move count. `total_score` just adds the three components —
TRL logs each one separately, so a run that stalls shows *which* component
stopped improving.

In [ ]:
def solved_score(stats: dict) -> float:
    """Sparse bonus for solving, plus a bounded penalty for rambling past the optimum."""
    extra_moves = max(0, stats["total_moves"] - stats["optimal_moves"])
    score = -OVERSHOOT_WEIGHT * min(1.0, extra_moves / (OVERSHOOT_TOLERANCE * stats["optimal_moves"]))
    if stats["solved"]:
        score += SOLVED_BONUS
        if stats["total_moves"] == stats["optimal_moves"]:
            score += OPTIMAL_BONUS
    return score


def total_score(stats: dict) -> float:
    return format_score(stats) + progress_score(stats) + solved_score(stats)

TRL forwards every extra dataset column to the reward function as a
kwarg, so `n_disks`/`source`/`auxiliary`/`target` arrive as lists aligned
with `completions`, and it names each logged reward column after the
function it came from — hence three thin callbacks around `batch_stats`
rather than one reward function.

In [ ]:
def batch_stats(completions, n_disks, source, auxiliary, target) -> List[dict]:
    return [
        reward_stats(completion_text(c),
                     {"n_disks": n, "source": src, "auxiliary": aux, "target": dst})
        for c, n, src, aux, dst in zip(completions, n_disks, source, auxiliary, target)
    ]


def format_reward(completions, n_disks, source, auxiliary, target, **kwargs):
    return [format_score(s) for s in batch_stats(completions, n_disks, source, auxiliary, target)]


def progress_reward(completions, n_disks, source, auxiliary, target, **kwargs):
    return [progress_score(s) for s in batch_stats(completions, n_disks, source, auxiliary, target)]


def solved_reward(completions, n_disks, source, auxiliary, target, **kwargs):
    return [solved_score(s) for s in batch_stats(completions, n_disks, source, auxiliary, target)]


REWARD_FUNCS = [format_reward, progress_reward, solved_reward]

## Learning from a reward instead of an answer

Everything so far needed a correct answer for every input. That is the
expensive part: someone has to write those answers, and the model can
never get better than they are.

Reinforcement learning (RL) needs something weaker, and often much easier
to get: a way to *score* whatever the model produced. Tower of Hanoi has
a verifier — replay the moves and look at where the disks ended up — so
we can score any attempt without knowing the right answer in advance.
That is the whole reason RL can pass the demonstrations it started from,
while supervised fine-tuning cannot.

Four words first, because the papers use them constantly:

* **policy** — the model, viewed as the thing that picks the next token.
  Training the model is "improving the policy".
* **rollout** — one complete attempt, generated by the policy.
* **reward** — a single number scoring one rollout. Each notebook's
  reward cell below shows exactly how ours is computed.
* **advantage** — how much better a rollout was than the alternatives.
  This, not the raw reward, is what the size of the update is
  proportional to: a reward of 0.9 tells you nothing until you know
  whether the other attempts got 0.1 or 0.95.

### Four algorithms, and which one we run

| Method | Full name | Year | From | Needs a value network? |
|---|---|---|---|---|
| **PPO** | Proximal Policy Optimization | 2017 | OpenAI | Yes |
| **GRPO** | Group Relative Policy Optimization | 2024 | DeepSeek | No |
| **GSPO** | Group Sequence Policy Optimization | 2025 | Qwen | No |
| **DAPO** | Decoupled Clip and Dynamic sAmpling PO | 2025 | ByteDance | No |

**PPO** improves the policy while stopping any single update from moving
it too far — too far being where training falls apart. It does that by
capping how much one step is allowed to change the policy's probabilities,
which is what everyone means by **clipping**. Its cost is a second
network, a *critic*, trained alongside the policy to predict how good a
situation is so that advantages can be worked out. It is expensive to
train and its predictions are noisy.

**GRPO** deletes the critic. For each prompt it samples a **group** of $G$
rollouts (8, in our config), scores all of them, and computes each one's
advantage by comparison against its own group:

$$
\hat{A}_i = \frac{r_i - \operatorname{mean}(r_1, \dots, r_G)}
                  {\operatorname{std}(r_1, \dots, r_G) + \varepsilon}
$$

($\varepsilon$ is a tiny constant so that a group where every rollout
scored the same does not divide by zero.) The group *is* the baseline: no
extra network to train, and nothing for it to be noisy about. That is why
GRPO is what most people reach for today.

**GSPO — what this notebook runs.** GSPO keeps the group-relative
advantage and changes exactly one thing. Both methods let a batch of
rollouts be reused for more than one gradient step, and after the first
step the policy has moved — so those rollouts came from a slightly
*older* policy than the one being updated. The correction is an
**importance ratio**: how much more (or less) likely the new policy is to
produce what the old one produced. (Use each batch for a single step and
every ratio is exactly 1, which is the case in which the two methods are
indistinguishable.) GRPO computes one such ratio per token; GSPO
computes a single length-normalised ratio for the whole sequence:

$$
s_i(\theta) = \exp\left(\frac{1}{|o_i|}\sum_{t=1}^{|o_i|}
  \log \frac{\pi_\theta(o_{i,t} \mid q, o_{i,<t})}
            {\pi_{\theta_\text{old}}(o_{i,t} \mid q, o_{i,<t})}\right)
$$

Here $\pi_\theta$ is the policy we are training, $\pi_{\theta_\text{old}}$
the one that generated the rollout $o_i$, and $|o_i|$ its length in
tokens. The reason to prefer this is granularity matching: our reward
scores a whole attempt with one number, so the update should weight the
whole attempt with one ratio. Per-token ratios against a whole-attempt
reward add variance without adding information.

![The GSPO loop: prompt, group of rollouts, reward each, normalise within the group, update](https://raw.githubusercontent.com/ivanvykopal/nlp-kinit-2026/main/images/gspo_loop.png)

*The group is the baseline: each rollout's advantage is its reward minus the group mean, over the group std — no critic required.*

See the token-level versus sequence-level importance-ratio figure in
[Zheng et al. (2025)](https://arxiv.org/abs/2507.18071) for the original
comparison this formula is drawn from; the paper is likewise under
arXiv's default non-exclusive licence, so we link to it rather than
reproducing the figure here.

### Three ways a reward goes wrong

Writing the reward is the actual work in RL, and these are the three
failures you will hit first.

**1. Reward hacking.** The model optimises the reward you wrote, not the
task you meant. Any component that grows with output length is paying the
model to ramble, and it will happily take the money: "+1 per legal move"
is maximised by shuffling disks legally forever and never solving
anything. The fix is not clever — it is to rank a handful of hand-written
attempts by hand and check the ordering comes out the way you intended,
which is what this notebook's reward cell does before any training runs.

**2. Reward sparsity.** If almost every early rollout scores zero, there
is nothing for the optimizer to climb: every attempt looks equally bad,
so no direction is preferred. A reward that pays only for a solved puzzle
has exactly this problem on a puzzle a fresh model never solves. Partial
credit — for disks correctly stacked in the flat notebook, for a single
move that got closer to the goal in the per-step one — turns a flat zero
into a slope the policy can walk up.

**3. Advantage, not reward.** Absolute reward values do not drive the
update; their position within the group does. Three rollouts scoring
around 0.9 and one scoring 0.1 produce an update that mostly says "not
that one" — the three good ones end up barely distinguished from each
other. Two consequences: a group where *every* rollout scores the same
contributes nothing at all (watch `reward_std` and
`frac_reward_zero_std` in the training logs), and a reward that is
generous to everybody is as useless as one that is generous to nobody.

### Two things about TRL that would otherwise confuse you

1. **The class below is called `GRPOTrainer` even though we are running
   GSPO.** At the TRL versions these notebooks were written against there
   is no separate GSPO trainer and no `loss_type="gspo"`; GSPO is what you
   get by setting `importance_sampling_level="sequence"`
   on the GRPO trainer — that one argument is the difference described
   above. Our own names say GSPO because that is what runs; `GRPOTrainer`
   and `GRPOConfig` are TRL's names, and we keep them as they are so the
   code matches TRL's documentation.
2. **A bare `GRPOConfig()` is not necessarily vanilla GRPO.** As of
   recent TRL versions it defaults to DAPO's loss formulation with the KL
   penalty — the term that holds the policy close to the model you started
   from — switched off, which is why no reference model gets loaded at all.
   And it is only *partial* DAPO: `epsilon_high` — the upward half of the
   asymmetric cap described above — is yours to set if you want it, as is
   `mask_truncated_completions=True` if
   you do not want to train on rollouts that were cut off mid-sentence.
   Defaults here move between releases, so check them against the version
   you have installed rather than trusting this paragraph; the point that
   survives is to read the config below as a list of deliberate choices.

## 1. Data

GSPO uses **prompts only** — there are no target trajectories in this
dataset at all, which is why the corrupted demonstrations cannot hurt it.

In [ ]:
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import GRPOConfig, GRPOTrainer

METHOD = "GSPO"
OUTPUT_DIR = RESULTS_DIR / "gspo_model"
INIT_FROM = "ivykopal/nlp-2026-hanoi-sft"
set_seed(SEED)
print("CUDA available:", torch.cuda.is_available())

if INIT_FROM.startswith(str(RESULTS_DIR)) and not Path(INIT_FROM).exists():
    raise FileNotFoundError(
        f"{INIT_FROM} not found -- run the FFT notebook first, "
        f"or set INIT_FROM = MODEL_NAME to start from the base model."
    )
print(f"policy will be initialised from: {INIT_FROM}")

gspo_rows = load_split("grpo_train", DATASET_REPO)
eval_splits = {
    "heldout": load_split("heldout", DATASET_REPO),
    "train_instances": load_split("train_instances", DATASET_REPO),
    "extrapolation": load_split("extrapolation", DATASET_REPO),
}
gspo_dataset = gspo_rows.map(to_gspo_format, remove_columns=gspo_rows.column_names)
print(f"GSPO prompts: {len(gspo_dataset)}")
print(f"disk counts: {sorted({r['n_disks'] for r in gspo_rows})}")

## 2. The reward

Three components, logged separately by TRL so a stalled run shows *which*
one stopped improving. `format_reward` penalises output that isn't a clean
list of moves; `progress_reward` gives dense partial credit for disks
correctly stacked, minus a flat penalty if an illegal move ended the
attempt; `solved_reward` gives +1.0 for solving, +0.5 more for solving in
the minimum number of moves, and a small penalty for rambling past the
optimum.

### Pitfall 1, on our own reward: reward hacking

The theory section already named the fix for reward hacking: rank a
handful of hand-written attempts and confirm the ordering matches what
you meant. The table below is that check, run on this task's own reward.
Watch the `legal but pointless (40 moves)` row in particular: forty legal
moves that solve nothing must score *below* a short correct answer, or
the reward is paying the model to shuffle disks forever.

In [ ]:
probe_sample = {"n_disks": 3, "source": "A", "auxiliary": "B", "target": "C"}
optimal_text = moves_to_text(solve_hanoi(3, "A", "B", "C"))
candidates = {
    "empty output": "",
    "prose, no moves": "I would need to think about this.",
    "legal but pointless (40 moves)": "A->B\n" + "B->C\nC->B\n" * 20,
    "illegal moves": "A->B\nA->B\nA->B\nA->B\n",
    "partial progress": "\n".join(optimal_text.splitlines()[:4]),
    "solved, wasteful": optimal_text + "\nC->B\nB->C",
    "solved optimally": optimal_text,
}
print(f"{'trajectory':<34}{'reward':>8}  {'solved':>7}{'moves':>7}")
print("-" * 60)
for label, text in candidates.items():
    stats = reward_stats(text, probe_sample)
    print(f"{label:<34}{total_score(stats):>8.2f}  {str(stats['solved']):>7}{stats['total_moves']:>7}")

### Pitfall 2, on our own reward: sparsity

The theory section claimed that a solve-only reward is flat, and therefore
useless early in training. Here is that claim measured on three prefixes of
the optimal solution, none of which finishes the puzzle. `solved_score` —
the sparse half of our reward — gives all three exactly the same number, so
on its own it offers the optimizer no reason to prefer any of them. Adding
`progress_score`, which pays partial credit per disk correctly stacked on
the target peg, turns that flat line into a slope.

In [ ]:
attempts = {
    "2 moves in": "\n".join(optimal_text.splitlines()[:2]),
    "4 moves in": "\n".join(optimal_text.splitlines()[:4]),
    "6 moves in": "\n".join(optimal_text.splitlines()[:6]),
}
print(f"{'attempt':<16}{'solved-only':>13}{'with progress':>15}")
print("-" * 44)
for label, text in attempts.items():
    stats = reward_stats(text, probe_sample)
    sparse_only = solved_score(stats) + 0.0  # + 0.0 so -0.0 prints as 0.00
    print(f"{label:<16}{sparse_only:>13.2f}"
          f"{sparse_only + progress_score(stats):>15.2f}")

### Pitfall 3, on four numbers: advantage, not reward

The theory section above (pitfall 3, "advantage, not reward") explains
why this matters; here it is in four lines of code.

In [ ]:
import numpy as np

rewards = np.array([0.9, 0.85, 0.95, 0.1])
advantage = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
for r, a in zip(rewards, advantage):
    print(f"reward {r:>5.2f}  ->  advantage {a:>+6.2f}")

## 3. GSPO configuration

The theory section above already explains what GSPO is and why each
mechanism exists; this cell only covers what is specific to *this run* —
change any of these and you also change what the run measures.

* `bf16=False, fp16=False` — a T4 cannot do bf16, and TRL's `bf16` default
  turns itself on when `fp16` is left unset, so leaving either one out
  crashes the run.
* `mask_truncated_completions=True` — TRL defaults this to `False`, which
  trains on rollouts that were cut off mid-sentence by the length limit;
  combined with a reward that already has to guard against length (Pitfall
  1 above), that is an easy way to reward-hack.
* `beta=0.0` — no KL penalty and no reference model, which saves the
  memory and the forward pass a reference model would otherwise cost.
  That is fine for a run this short, because the format reward already
  anchors the output shape; a longer run would want the KL term back to
  stop the policy drifting into degenerate text.
* `num_generations=8` with a per-device batch of 4 and 4 gradient
  accumulation steps gives 2 prompts x 8 rollouts = 16 completions per
  optimizer step. Shrink `num_generations` and the group gets too small
  for the group-relative advantage (theory section) to mean anything.
* `loss_type="grpo"` with `importance_sampling_level="sequence"` is what
  makes this GSPO rather than plain GRPO — why sequence-level, see the
  theory section above.
* `num_iterations=2` reuses each rollout batch for two gradient steps
  before drawing a fresh one, which is why `GSPO_MAX_STEPS` is doubled to
  120 (see `config.py`): at one gradient step per batch, 120 steps would
  mean 120 fresh rollout batches, but at two steps per batch it is only
  60 — doubling the step count keeps the number of rollouts the policy
  actually sees the same either way.

In [ ]:
GSPO_KWARGS = dict(
    max_completion_length=GSPO_MAX_COMPLETION_LENGTH,
    per_device_train_batch_size=GSPO_BATCH_SIZE,
    gradient_accumulation_steps=GSPO_GRAD_ACCUM,
    num_generations=GSPO_NUM_GENERATIONS,
    learning_rate=GSPO_LEARNING_RATE,
    beta=0.0,
    temperature=1.0,
    mask_truncated_completions=True,
    max_grad_norm=0.5,
    logging_steps=5,
    save_strategy="no",
    bf16=False,
    fp16=False,
    report_to="none",
    seed=SEED,
    log_completions=True,
    num_completions_to_print=2,
    loss_type="grpo",
    importance_sampling_level="sequence",
    num_iterations=1,
)

## 4. Attempt one: cold start from the base model

This run is here to fail. GSPO learns from *differences within a group*:
if all 8 rollouts for a prompt score about the same, every advantage is
~0 and no gradient reaches the policy. Watch `reward_std` (spread within
a group — near zero means no signal) and `frac_reward_zero_std` (fraction
of groups where every rollout scored identically — close to 1.0 means the
run is dead no matter how many steps you give it).

In [ ]:
RUN_COLD_START = True

if RUN_COLD_START:
    cold_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
    cold_model.config.use_cache = False
    cold_model.to("cuda" if torch.cuda.is_available() else "cpu")
    print(f"cold_model on: {next(cold_model.parameters()).device}")
    cold_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if cold_tokenizer.pad_token is None:
        cold_tokenizer.pad_token = cold_tokenizer.eos_token

    cold_trainer = GRPOTrainer(
        model=cold_model, reward_funcs=REWARD_FUNCS,
        args=GRPOConfig(output_dir=str(RESULTS_DIR / "gspo_cold_start"),
                         max_steps=GSPO_COLD_START_MAX_STEPS, **GSPO_KWARGS),
        train_dataset=gspo_dataset, processing_class=cold_tokenizer,
    )
    cold_trainer.train()

    cold_history = plotting.history_from_log(cold_trainer.state.log_history)
    plotting.plot_history(cold_history, ["reward", "reward_std", "frac_reward_zero_std"],
                           RESULTS_DIR / "gspo_cold_start_reward.png",
                           title="GSPO from base model: no reward spread, no gradient",
                           ylabel="reward")

    del cold_model, cold_trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Attempt two: GSPO on top of the fine-tuned checkpoint

Same algorithm, same reward, same data. The only change is the starting
policy: one that already produces well-formed move lists and solves
maybe half the puzzles, so a group of 8 rollouts contains both successes
and failures — `reward_std` is non-zero, and there is a gradient to
follow. This is also why real pipelines are SFT-then-RL rather than RL
alone: SFT buys the exploration RL needs.

### What evaluation needs

Evaluating a model on this task takes exactly two task-specific functions:
one that turns a puzzle into a chat prompt, and one that scores whatever
the model wrote. Bundling them means every evaluation call below fits on
one line.

In [ ]:
TASK = evaluation.FlatTask(compute_stats=compute_stats, to_chat_prompt=to_chat_prompt)

### Set up the policy to train

Loads the FFT checkpoint (not the base model — see section 4 for why),
and wires up the mid-training probe: a small closed-loop check on a fixed
subset of held-out puzzles, run periodically so training progress is
visible before the run finishes.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(INIT_FROM)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(INIT_FROM, dtype=torch.float32)
model.config.use_cache = False
model.to("cuda" if torch.cuda.is_available() else "cpu")
print(f"model on: {next(model.parameters()).device}")
print(f"initialised from {INIT_FROM}")

probe_samples = [row for row in eval_splits["heldout"] if row["n_disks"] in PROBE_GROUP_VALUES]
probe_callback = probe.MetricProbe(probe_samples, tokenizer, TASK,
                                    every=GSPO_PROBE_EVERY,
                                    max_new_tokens=EVAL_MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE)
print(f"probe subset: {len(probe_samples)} held-out instances, disks {PROBE_GROUP_VALUES}")

gspo_trainer = GRPOTrainer(
    model=model, reward_funcs=REWARD_FUNCS,
    args=GRPOConfig(output_dir=str(OUTPUT_DIR), max_steps=GSPO_MAX_STEPS, **GSPO_KWARGS),
    train_dataset=gspo_dataset, processing_class=tokenizer, callbacks=[probe_callback],
)
gspo_trainer.train()

Save the checkpoint for the three-way comparison below.

In [ ]:
gspo_trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"saved to {OUTPUT_DIR}")

## 6. Reading the RL curves

`reward` should climb. `reward_std` should stay above zero — the learning
signal. `frac_reward_zero_std` should be well below 1.0, unlike the
cold-start run. The three reward components separate *what* improved:
`format_reward` rising to ~0 means clean output shape, `solved_reward`
rising means puzzles are actually being finished.

In [ ]:
gspo_history = plotting.history_from_log(gspo_trainer.state.log_history)
plotting.plot_history(gspo_history, ["reward", "reward_std", "frac_reward_zero_std"],
                       RESULTS_DIR / "gspo_reward.png", title="GSPO from fine-tuned checkpoint",
                       ylabel="reward")
plotting.plot_history(gspo_history, ["completions/clipped_ratio", "completions/mean_length"],
                       RESULTS_DIR / "gspo_lengths.png", title="GSPO: rollout lengths and truncation",
                       ylabel="value")
plotting.plot_history(
    gspo_history,
    ["rewards/format_reward/mean", "rewards/progress_reward/mean", "rewards/solved_reward/mean"],
    RESULTS_DIR / "gspo_reward_components.png", title="GSPO: reward components", ylabel="reward",
)
probe_callback.plot(RESULTS_DIR / "gspo_probe.png",
                     title=f"GSPO: solved rate on {len(probe_samples)} held-out instances")

## 7. Evaluation

Score the trained policy on all four splits, the same way the FFT and
LoRA notebooks did, so the numbers below are directly comparable.

In [ ]:
gspo_report = evaluation.evaluate_model(model, tokenizer, eval_splits, METHOD, TASK,
                                         group_key=PROBE_GROUP_KEY,
                                         max_new_tokens=EVAL_MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE)
report.print_report(gspo_report)
report.save_report(gspo_report, RESULTS_DIR)

## 9. A single prediction

One 4-disk held-out puzzle in full, the same way the FFT and LoRA
notebooks end, so you can put the three outputs side by side.

In [ ]:
heldout = eval_splits["heldout"]
index = next(i for i, r in enumerate(heldout) if r["n_disks"] == 4)
report.print_example(gspo_report["splits"]["heldout"]["completions"][index], heldout[index],
                      compute_stats)

## 10. What to take away

* **FFT and LoRA land in the same place.** They optimise the same
  objective on the same imperfect data. PEFT buys memory and disk, not
  accuracy.
* **GSPO can pass both**, because it optimises a verifiable reward
  instead of imitating demonstrations. When you have a checker — a
  compiler, a unit test, a puzzle simulator, a proof assistant — you are
  not limited by the quality of your labels.
* **RL is not a replacement for SFT.** The cold-start run in section 4 is
  what happens without one.
* **Reward design is the actual work.** Section 2's table is a
  two-minute check that catches a reward paying the model to shuffle
  disks forever. Always rank a handful of hand-written trajectories
  before launching a run.